# upload

Pushes the trained model to `notoh/qwen-coder-compilers` under
`{parameter-size}/{dataset-size}/`. Standalone — only requires `.env` and a completed `train.ipynb` run.

In [ ]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import HfApi

PROJECT_ROOT = Path("../..").resolve()
load_dotenv(PROJECT_ROOT / ".env")
MODEL = os.environ["MODEL"]
SEED  = int(os.environ["SEED"])

pattern  = f"{MODEL.split('/')[-1]}_seed{SEED}_*"
matches  = list((PROJECT_ROOT / "models").glob(pattern))
assert len(matches) == 1, f"expected 1 match for {pattern}, got {matches}"
OUT_DIR  = matches[0]

parts        = OUT_DIR.name.rsplit("_", 1)
dataset_size = parts[-1]
param_size   = re.search(r'\d+\.?\d*[BbMm]', parts[0])
param_size   = param_size.group(0).upper() if param_size else parts[0]

hf_path  = f"{param_size}/{dataset_size}"
HUB_REPO = "notoh/qwen-coder-compilers"

print(f"uploading {OUT_DIR.name} -> {HUB_REPO}/{hf_path}")
HfApi().upload_folder(folder_path=OUT_DIR, repo_id=HUB_REPO, path_in_repo=hf_path)
print("done")